## Import

In [1]:
from bertopic import BERTopic
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import csv
from evaluation import evaluate_model
from bertopic.vectorizers import ClassTfidfTransformer

df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])

c:\Users\aleblu\AppData\Local\miniconda3\envs\NLP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 32/32 [00:13<00:00,  2.44it/s]


In [29]:
run_name = "BERTopic_nr_topics_5"

# Hyperparameters

n_neighbors = 30 # 15 -> BEST 30
n_components = 5 # 5 -> BEST 5
random_state = [0, 1, 37, 42, 73]
min_dist = 0.0
min_cluster_size = 10 # 10 -> BEST 10
min_df = 2 # 2 -> BEST 2 BUT 1 GOOD
ngram_range = (1, 2) # (1, 2) -> BEST (1, 2)
top_n_words = 10 # 10 -> BEST 10
nr_topics = 5 # auto

# Data saving

data = []

# Setup different models

hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", min_df=min_df, ngram_range=ngram_range)
# ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True) # enable by default --> BEST DEFAULT

# Train

for seed in random_state:
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=0.0, metric='cosine', random_state=seed)

    topic_model = BERTopic(

        # Pipeline models
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        #ctfidf_model=ctfidf_model,

        # Hyperparameters
        top_n_words=top_n_words,
        n_gram_range=ngram_range,
        min_topic_size="auto", #use HDBSCAN
        verbose=True,
        nr_topics = nr_topics, # base is auto

        # General parameters
        calculate_probabilities=True,
        language="english"
    )

    topics, probs = topic_model.fit_transform(docs, embeddings)

    # Evaluate model

    scores = evaluate_model(topic_model, docs, topk=topic_model.top_n_words)
    
    data.append([seed, scores[0], scores[1]])

df = pd.DataFrame(data, columns=["seed", "Diversity", "Coherence"])
df

2025-01-30 15:11:27,389 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-01-30 15:11:29,932 - BERTopic - Dimensionality - Completed ✓
2025-01-30 15:11:29,934 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-01-30 15:11:29,983 - BERTopic - Cluster - Completed ✓
2025-01-30 15:11:29,985 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-01-30 15:11:30,051 - BERTopic - Representation - Completed ✓
2025-01-30 15:11:30,052 - BERTopic - Topic reduction - Reducing number of topics
2025-01-30 15:11:30,113 - BERTopic - Topic reduction - Reduced number of topics from 17 to 5
100%|██████████| 1/1 [00:00<00:00,  3.11it/s]
2025-01-30 15:11:33,075 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-01-30 15:11:35,411 - BERTopic - Dimensionality - Completed ✓
2025-01-30 15:11:35,412 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-01-30 15:11:35,465 - BERTop

,seed,Diversity,Coherence
0,0,0.72,0.049552
1,1,0.70,0.048950
2,37,0.68,0.022055
3,42,0.68,0.048875
4,73,0.70,0.064665


Save model

In [30]:
embedding_model = "all-MiniLM-L6-v2"
topic_model.save(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", run_name), serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

Save data

In [31]:
param_str = str({
    "n_neighbors": n_neighbors,
    "n_components": n_components,
    "min_dist": min_dist,
    "random_state": random_state,
    "min_cluster_size": min_cluster_size,
    "min_df": min_df,
    "ngram_range": ngram_range,
    "top_n_words": top_n_words
})


data_run = [[run_name, param_str, df["Diversity"].mean(), df["Coherence"].mean()]]

df_run = pd.DataFrame(data_run, columns=["run_name", "params", "diversity", "coherence"])

df_run.to_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", "results.csv"), mode="a", header=False, index=False)

Show results

In [32]:
df_results = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", "results.csv"))
df_results

,run_name,params,diversity,coherence
0,base,"{'n_neighbors': 15, 'n_components': 5, 'min_di...",0.518766,0.018410
1,UMAP_n_neighbors_20,"{'n_neighbors': 20, 'n_components': 5, 'min_di...",0.508158,0.015145
2,UMAP_n_neighbors_25,"{'n_neighbors': 25, 'n_components': 5, 'min_di...",0.539043,0.011082
3,UMAP_n_neighbors_30,"{'n_neighbors': 30, 'n_components': 5, 'min_di...",0.543632,0.024256
4,UMAP_n_neighbors_35,"{'n_neighbors': 35, 'n_components': 5, 'min_di...",0.533987,0.014548
5,UMAP_n_neighbors_40,"{'n_neighbors': 40, 'n_components': 5, 'min_di...",0.529626,0.021350
6,UMAP_n_neighbors_45,"{'n_neighbors': 45, 'n_components': 5, 'min_di...",0.537632,0.020211
7,UMAP_n_neighbors_50,"{'n_neighbors': 50, 'n_components': 5, 'min_di...",0.520158,0.022319
8,UMAP_n_neighbors_55,"{'n_neighbors': 55, 'n_components': 5, 'min_di...",0.528634,0.025503
9,UMAP_n_neighbors_60,"{'n_neighbors': 60, 'n_components': 5, 'min_di...",0.532111,0.023156


In [33]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,312,-1_gnome_gnomes_mushrooms_yellow,"[gnome, gnomes, mushrooms, yellow, red, colour...",[There are 2 teams of gnomes - The yellow bask...
1,0,541,0_basket_gnomes_mushrooms_red,"[basket, gnomes, mushrooms, red, gnome, yellow...",[There are different colour of gnomes with a t...
2,1,97,1_points_hats_blue_pink,"[points, hats, blue, pink, tall, brown, colour...","[It's quite hard to discern a pattern, but I f..."
3,2,31,2_keys_just_game_fingers,"[keys, just, game, fingers, breaks, make, sure...",[You really just have to go with your intuitio...
4,3,19,3_pattern_random_advice_did,"[pattern, random, advice, did, just, game, luc...","[I found no discernable pattern, it was almost..."


In [34]:
topic_model.visualize_hierarchy(top_n_topics=50)

In [35]:
topic_model.visualize_documents(docs)